# 8.3 重塑和透视

## 8.3.1 使用层次化索引进行重塑

In [55]:
# stack unstack

In [56]:
import pandas as pd
import numpy as np
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(["Ohio", "Colorado"], name="state"),
                    columns=pd.Index(["one", "two", "three"],
                    name="number"))
data

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [57]:
result = data.stack()  # 转成层次化索引
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int64

In [58]:
result.unstack()

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


In [59]:
s1 = pd.Series([0, 1, 2, 3], index=["a", "b", "c", "d"], dtype="Int64")
s2 = pd.Series([4, 5, 6], index=["c", "d", "e"], dtype="Int64")
data2 = pd.concat([s1, s2], keys=["one", "two"])
data2

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: Int64

In [60]:
data2.unstack()

,a,b,c,d,e
one,0,1,2,3,<NA>
two,<NA>,<NA>,4,5,6


In [61]:
# stack操作会默认过滤NA 所以该运算可逆
data2.unstack().stack()

one  a       0
     b       1
     c       2
     d       3
     e    <NA>
two  a    <NA>
     b    <NA>
     c       4
     d       5
     e       6
dtype: Int64

## 8.3.2 将长格式透视为宽格式

In [62]:
data = pd.read_csv('../examples/macrodata.csv')
data = data[['year', 'quarter', 'realgdp', 'infl', 'unemp']]  # 只选列
data.head()

,year,quarter,realgdp,infl,unemp
0,1959,1,2710.349,0.00,5.8
1,1959,2,2778.801,2.34,5.1
2,1959,3,2775.488,2.74,5.3
3,1959,4,2785.204,0.27,5.6
4,1960,1,2847.699,2.31,5.2


In [63]:
years = data.pop('year').values
print(type(years))
quarters = data.pop('quarter').values
print(type(quarters))
periods = pd.PeriodIndex.from_fields(year=years, quarter=quarters)
periods.name = 'date'
periods

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


PeriodIndex(['1959Q1', '1959Q2', '1959Q3', '1959Q4', '1960Q1', '1960Q2',
             '1960Q3', '1960Q4', '1961Q1', '1961Q2',
             ...
             '2007Q2', '2007Q3', '2007Q4', '2008Q1', '2008Q2', '2008Q3',
             '2008Q4', '2009Q1', '2009Q2', '2009Q3'],
            dtype='period[Q-DEC]', name='date', length=203)

In [64]:
data.index = periods.to_timestamp('D')
data.head()

,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


In [65]:
data.columns=['realgdp', 'infl', 'unemp']
data.columns.name = 'item'
data.head()

item,realgdp,infl,unemp
date,,,
1959-01-01,2710.349,0.00,5.8
1959-04-01,2778.801,2.34,5.1
1959-07-01,2775.488,2.74,5.3
1959-10-01,2785.204,0.27,5.6
1960-01-01,2847.699,2.31,5.2


In [66]:
data1 = data.stack()
data1

date        item   
1959-01-01  realgdp     2710.349
            infl           0.000
            unemp          5.800
1959-04-01  realgdp     2778.801
            infl           2.340
                         ...    
2009-04-01  infl           3.370
            unemp          9.200
2009-07-01  realgdp    12990.341
            infl           3.560
            unemp          9.600
Length: 609, dtype: float64

In [67]:
data2 = data1.reset_index()
data2

,date,item,0
0,1959-01-01,realgdp,2710.349
1,1959-01-01,infl,0.000
2,1959-01-01,unemp,5.800
3,1959-04-01,realgdp,2778.801
4,1959-04-01,infl,2.340
...,...,...,...
604,2009-04-01,infl,3.370
605,2009-04-01,unemp,9.200
606,2009-07-01,realgdp,12990.341
607,2009-07-01,infl,3.560


In [68]:
data2.rename(columns={0:'value'}, inplace=True)
data2.head()

,date,item,value
0,1959-01-01,realgdp,2710.349
1,1959-01-01,infl,0.000
2,1959-01-01,unemp,5.800
3,1959-04-01,realgdp,2778.801
4,1959-04-01,infl,2.340


In [69]:
pivoted = data2.pivot(index='date', columns='item', values='value')
pivoted.head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


In [70]:
data2['value2'] = np.random.randn(len(data2))
data2[:10]

,date,item,value,value2
0,1959-01-01,realgdp,2710.349,-0.389850
1,1959-01-01,infl,0.000,1.108958
2,1959-01-01,unemp,5.800,-0.541424
3,1959-04-01,realgdp,2778.801,1.431352
4,1959-04-01,infl,2.340,-0.579804
5,1959-04-01,unemp,5.100,-1.190449
6,1959-07-01,realgdp,2775.488,0.480455
7,1959-07-01,infl,2.740,-0.870123
8,1959-07-01,unemp,5.300,0.403016
9,1959-10-01,realgdp,2785.204,1.121494


In [71]:
pivoted = data2.pivot(index='date', columns='item')
pivoted.head()

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8  1.108958 -0.389850 -0.541424
1959-04-01  2.34  2778.801   5.1 -0.579804  1.431352 -1.190449
1959-07-01  2.74  2775.488   5.3 -0.870123  0.480455  0.403016
1959-10-01  0.27  2785.204   5.6 -0.819339  1.121494  2.148402
1960-01-01  2.31  2847.699   5.2 -1.819875  1.163030  1.358631

In [72]:
pivoted['value'].head()

item,infl,realgdp,unemp
date,,,
1959-01-01,0.00,2710.349,5.8
1959-04-01,2.34,2778.801,5.1
1959-07-01,2.74,2775.488,5.3
1959-10-01,0.27,2785.204,5.6
1960-01-01,2.31,2847.699,5.2


In [74]:
# pivot相当于set_index + unstack
unstacked = data2.set_index(['date', 'item']).unstack(level='item')
unstacked.head()

value                    value2                    
item        infl   realgdp unemp      infl   realgdp     unemp
date                                                          
1959-01-01  0.00  2710.349   5.8  1.108958 -0.389850 -0.541424
1959-04-01  2.34  2778.801   5.1 -0.579804  1.431352 -1.190449
1959-07-01  2.74  2775.488   5.3 -0.870123  0.480455  0.403016
1959-10-01  0.27  2785.204   5.6 -0.819339  1.121494  2.148402
1960-01-01  2.31  2847.699   5.2 -1.819875  1.163030  1.358631

## 8.3.3 将宽格式透视为长格式

In [75]:
df = pd.DataFrame({"key": ["foo", "bar", "baz"],
                   "A": [1, 2, 3],
                   "B": [4, 5, 6],
                   "C": [7, 8, 9]})
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


In [76]:
melted = pd.melt(df, id_vars="key")
melted

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6
6,foo,C,7
7,bar,C,8
8,baz,C,9


In [77]:
reshaped = melted.pivot(index="key", columns="variable",
                        values="value")
reshaped

variable,A,B,C
key,,,
bar,2,5,8
baz,3,6,9
foo,1,4,7


In [78]:
reshaped.reset_index()

variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7


In [79]:
pd.melt(df, id_vars="key", value_vars=["A", "B"])

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6


In [80]:
pd.melt(df, value_vars=["A", "B", "C"])
pd.melt(df, value_vars=["key", "A", "B"])

,variable,value
0,key,foo
1,key,bar
2,key,baz
3,A,1
4,A,2
5,A,3
6,B,4
7,B,5
8,B,6


# End